<h2 style="color:blue;">Importation des bibliothèques</h2>
Ce bloc regroupe l'ensemble des bibliothèques nécessaires au projet, couvrant la manipulation des données, le machine learning et le deep learning.
- <span style="color:red">Gestion des données</span>
  - **sqlite3 :** interaction avec une base de données SQLite
  - **pandas :** manipulation des données sous forme de DataFrame
  - **numpy :** calculs numériques et tableaux
- <span style="color:red">Sauvegarde:</span>
  - **joblib :** sauvegarde et chargement de modèles
- <span style="color:red">Deep Learning (TensorFlow / Keras):</span> 
  - **Sequential :** création du modèle
  - **Dense, Dropout :** couches du réseau de neurones
  - **to_categorical :** encodage des labels
- <span style="color:red">Machine Learning (Scikit-learn)::</span>-
  - **train_test_split :** séparation des données
  - **LabelEncoder :** conversion des labels en valeurs numériques






In [1]:
import sqlite3
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

2026-04-19 22:49:47.741089: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-19 22:49:53.588980: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-19 22:49:57.931902: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-19 22:50:01.607797: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-19 22:50:02.524017: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-19 22:50:05.909932: I tensorflow/core/platform/cpu_feature_gu

<h2 style="color:blue;">Chargement du vectoriseur et des données</h2>

Ce bloc permet de charger le vectoriseur **TF-IDF** déjà entraîné ainsi que les données textuelles depuis une base de données SQLite, afin de préparer le traitement des textes.

- <span style="color:red">Définition des chemins :</span>
  - **DB_PATH :** chemin vers la base de données contenant les documents
  - **PKL_VECTORIZER :** fichier contenant le vectoriseur TF-IDF sauvegardé
- <span style="color:red">Chargement du vectoriseur :</span>
  - **joblib.load :** permet de charger le vectoriseur TF-IDF préalablement entraîné
- <span style="color:red">Chargement des données :</span>
  - **sqlite3.connect :** connexion à la base de données SQLite
  - **read_sql_query :** récupération des colonnes clean_text et type depuis la table documents
- <span style="color:red">Nettoyage des données :</span>
  - **fillna("") :** remplace les valeurs manquantes dans les textes par une chaîne vide pour éviter les erreurs lors du traitement

In [2]:
DB_PATH = "../Data/dataset.db"
PKL_VECTORIZER = "../models/tfidf_vectorizer.pkl"

print("Chargement du vectoriseur et des données...")
tfidf = joblib.load(PKL_VECTORIZER)

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT clean_text, type FROM documents", conn)
conn.close()

df['clean_text'] = df['clean_text'].fillna("")

Chargement du vectoriseur et des données...


<h2 style="color:blue;">Encodage et préparation des données</h2>
Ce bloc prépare les données pour l'entraînement du modèle de classification en effectuant l'encodage des labels, la vectorisation des textes et la séparation en données d'entraînement et de test.

- **LabelEncoder :** transforme les classes (type) en valeurs numériques
- **to_categorical :** convertit les labels en format one-hot (multi-classes)
- **tfidf.transform :** transforme les textes en matrice TF-IDF (format sparse, optimisé mémoire)
- **train_test_split :**
  - 80% entraînement / 20% test
  - stratify : conserve la distribution des classes
  - random_state=42 : résultats reproductibles


In [3]:
le = LabelEncoder()
y_encoded = le.fit_transform(df['type'])
y_categorical = to_categorical(y_encoded)

# On transforme tout le texte en matrice Creuse (Sparse) - Cela ne sature PAS la RAM
print("Conversion en matrice Sparse (Légère)...")
X_sparse = tfidf.transform(df['clean_text'])

# Division Train/Test
X_train_sparse, X_test_sparse, y_train, y_test = train_test_split(
    X_sparse, y_categorical, test_size=0.2, random_state=42, stratify=y_encoded
)

Conversion en matrice Sparse (Légère)...


<h2 style="color:blue;">Générateur de batchs</h2>

Ce générateur permet d'entraîner le modèle sans charger toute la matrice en mémoire.

- Mélange les données à chaque époque
- Extrait les données par petits lots (batchs)
- Convertit uniquement le batch courant en format dense
- Réduit fortement l'utilisation de la RAM

In [4]:
def batch_generator(X_sparse, y_labels, batch_size):
    """
    Générateur qui convertit de petites parties de la matrice 
    en format Dense uniquement au moment du passage dans le réseau.
    """
    number_of_samples = X_sparse.shape[0]
    while True:
        # Mélanger les indices à chaque époque pour un meilleur apprentissage
        indices = np.arange(number_of_samples)
        np.random.shuffle(indices)
        
        for start in range(0, number_of_samples, batch_size):
            end = min(start + batch_size, number_of_samples)
            batch_indices = indices[start:end]
            
            # Extraction et conversion en Dense + float32 (économique)
            x_batch = X_sparse[batch_indices].toarray().astype('float32')
            y_batch = y_labels[batch_indices]
            
            yield x_batch, y_batch

<h2 style="color:blue;">Construction et compilation du modèle</h2>

Ce réseau de neurones est utilisé pour classer les textes en deux catégories (médical / non-médical) à partir des vecteurs TF-IDF.

- <span style="color:red">Couche d’entrée (Dense 512):</span>
  - Reçoit un vecteur de taille 5000 (issu du TF-IDF)
  - Applique une activation ReLU pour introduire de la non-linéarité
  - Permet d’apprendre des combinaisons importantes de mots
- <span style="color:red">Dropout (0.3):</span>
  - Désactive aléatoirement 30% des neurones pendant l’entraînement
  - Réduit le sur-apprentissage (overfitting)
- <span style="color:red">Couche Dense (256 neurones)</span>
  - Apprend des représentations plus abstraites des données
  - Activation ReLU pour maintenir la capacité d’apprentissage
- <span style="color:red">Dropout (0.2):</span> 
  - Renforce la régularisation du modèle
- <span style="color:red">Couche de sortie (2 neurones):</span> 
  - Correspond aux 2 classes : médical / non-médical
  - Activation softmax pour obtenir des probabilités

In [5]:
model = Sequential([
    # Couche d'entrée (5000 neurones car max_features=5000)
    Dense(512, activation='relu', input_shape=(5000,)),
    Dropout(0.3), # Évite le sur-apprentissage
    
    Dense(256, activation='relu'),
    Dropout(0.2),
    
    # Couche de sortie (2 neurones : médical / non-médical)
    Dense(2, activation='softmax')
])
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/home/fateh/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


<h2 style="color:blue;">Entraînement du modèle</h2>

Ce bloc lance l'entraînement du réseau de neurones en utilisant un générateur de batchs pour éviter de surcharger la mémoire.

- <span style="color:red">Configuration de l'entraînement:</span> 
  - **BATCH_SIZE** = 64
     - nombre d'exemples traités à chaque itération
     - permet un bon équilibre entre vitesse et stabilité
  - **steps_per_epoch**
     - nombre de batchs par époque
     - calculé en divisant la taille du dataset d'entraînement par la taille d’un batch
- <span style="color:red">Préparation de la validation:</span> 
    - Le jeu de test est converti en matrice dense une seule fois
    - Utilisé pour évaluer les performances du modèle après chaque époque
- <span style="color:red">Entraînement:</span> 
  - **batch_generator :** fournit les données par petits lots
  - **epochs=20 :** le modèle voit tout le dataset 20 fois
  - **validation_data :** permet de suivre la performance sur des données jamais vues
  - **verbose=1 :** affiche la progression de l’entraînement




In [6]:
BATCH_SIZE = 64
steps_per_epoch = X_train_sparse.shape[0] // BATCH_SIZE

print(f"\n--- Début de l'entraînement par batchs ({BATCH_SIZE} docs à la fois) ---")

# On convertit le set de TEST une seule fois (si la RAM le permet) pour la validation
x_test_dense = X_test_sparse.toarray().astype('float32')

model.fit(
    batch_generator(X_train_sparse, y_train, BATCH_SIZE),
    steps_per_epoch=steps_per_epoch,
    epochs=20,
    validation_data=(x_test_dense, y_test),
    verbose=1
)


--- Début de l'entraînement par batchs (64 docs à la fois) ---
Epoch 1/20
1564/1565 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.9845 - loss: 0.0447

2026-04-19 22:55:53.417017: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 501000000 exceeds 10% of free system memory.


1565/1565 ━━━━━━━━━━━━━━━━━━━━ 72s 40ms/step - accuracy: 0.9845 - loss: 0.0446 - val_accuracy: 0.9992 - val_loss: 0.0025
Epoch 2/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 53s 34ms/step - accuracy: 0.9998 - loss: 5.0846e-04 - val_accuracy: 0.9992 - val_loss: 0.0024
Epoch 3/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 1.0000 - loss: 1.2036e-04 - val_accuracy: 0.9988 - val_loss: 0.0060
Epoch 4/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 53s 34ms/step - accuracy: 1.0000 - loss: 3.3383e-05 - val_accuracy: 0.9993 - val_loss: 0.0039
Epoch 5/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 1.0000 - loss: 1.2723e-06 - val_accuracy: 0.9994 - val_loss: 0.0038
Epoch 6/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 57s 37ms/step - accuracy: 1.0000 - loss: 2.1930e-07 - val_accuracy: 0.9994 - val_loss: 0.0039
Epoch 7/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 1.0000 - loss: 8.5408e-08 - val_accuracy: 0.9993 - val_loss: 0.0040
Epoch 8/20
1565/1565 ━━━━━━━━━━━━━━━━━━━━ 56s 36ms/step - accur

<h2 style="color:blue;">Sauvegarde du modèle et du label encoder</h2>

Ce bloc permet de sauvegarder les éléments essentiels du projet afin de les réutiliser sans réentraînement.

- Le modèle de deep learning est sauvegardé pour être rechargé plus tard
- Le LabelEncoder est également enregistré pour garantir la cohérence des classes lors de la prédiction
- Cette étape évite de refaire tout le processus d'entraînement

In [7]:
model.save("../models/medical_model_final.h5")
joblib.dump(le, "../models/label_encoder.pkl")
print("\nModèle sauvegardé avec succès sans saturer la RAM !")


Modèle sauvegardé avec succès sans saturer la RAM !
